In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("therohk/million-headlines")

print("Path to dataset files:", path)

100%|██████████| 21.4M/21.4M [00:00<00:00, 125MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/therohk/million-headlines/versions/5


In [14]:
data = pd.read_csv(f'{path}/abcnews-date-text.csv')
headlines = data['headline_text'].values[:10000]  # Use first 10,000 for sp

headlines[:10]

array(['aba decides against community broadcasting licence',
       'act fire witnesses must be aware of defamation',
       'a g calls for infrastructure protection summit',
       'air nz staff in aust strike for pay rise',
       'air nz strike to affect australian travellers',
       'ambitious olsson wins triple jump',
       'antic delighted with record breaking barca',
       'aussie qualifier stosur wastes four memphis match',
       'aust addresses un security council over iraq',
       'australia is locked into war timetable opp'], dtype=object)

In [5]:
# Create character vocabulary
text = ' '.join(headlines)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

In [6]:
# Prepare sequences (predict next character)
seq_length = 40
X, y = [], []
for i in range(0, len(text) - seq_length, 1):
    seq_in = text[i:i + seq_length]
    seq_out = text[i + seq_length]
    X.append([char_to_idx[c] for c in seq_in])
    y.append(char_to_idx[seq_out])

X = np.array(X)
y = np.array(y)

In [7]:
# Reshape X for RNN [samples, time steps, features]
X = np.reshape(X, (X.shape[0], seq_length, 1)) / float(len(chars))  # Normalize
y = np.eye(len(chars))[y]  # One-hot encode

# Split into training and testing
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [8]:
# Build RNN model
model = Sequential([
    SimpleRNN(50, activation='tanh', input_shape=(seq_length, 1)),
    Dense(len(chars), activation='softmax')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 50)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 37)             │         1,887 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,487 (17.53 KB)

 Trainable params: 4,487 (17.53 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.1, verbose=1)

# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

Epoch 1/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 31s 13ms/step - accuracy: 0.1785 - loss: 2.8670 - val_accuracy: 0.2095 - val_loss: 2.7296
Epoch 2/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - accuracy: 0.2060 - loss: 2.7185 - val_accuracy: 0.2124 - val_loss: 2.6971
Epoch 3/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 43s 13ms/step - accuracy: 0.2082 - loss: 2.6944 - val_accuracy: 0.2148 - val_loss: 2.6772
Epoch 4/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - accuracy: 0.2114 - loss: 2.6731 - val_accuracy: 0.2160 - val_loss: 2.6577
Epoch 5/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - accuracy: 0.2115 - loss: 2.6524 - val_accuracy: 0.2180 - val_loss: 2.6379
Epoch 6/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 41s 12ms/step - accuracy: 0.2142 - loss: 2.6389 - val_accuracy: 0.2179 - val_loss: 2.6270
Epoch 7/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 42s 13ms/step - accuracy: 0.2176 - loss: 2.6240 - val_accuracy: 0.2235 - val_loss: 2.6149
Epoch 8/10
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 39s 12ms/step - accuracy: 0.2211 -

In [11]:
# Generate new text
def generate_text(model, seed, length=100):
    generated = seed
    for _ in range(length):
        x = np.array([char_to_idx[c] for c in generated[-seq_length:]])
        x = x.reshape(1, seq_length, 1) / float(len(chars))
        pred = model.predict(x, verbose=0)
        next_idx = np.argmax(pred[0])
        generated += idx_to_char[next_idx]
    return generated

In [12]:
# Example generation
seed = headlines[0][:seq_length]
generated_text = generate_text(model, seed)
print(f'Generated Headline: {generated_text}')

Generated Headline: aba decides against community broadcasti so coash for tat to coash for tat to coash for tat to coash for tat to coash for tat to coash for t


In [15]:
# Example generation
seed = headlines[40][:seq_length]
generated_text = generate_text(model, seed)
print(f'Generated Headline: {generated_text}')

Generated Headline: direct anger at govt not soldiers crean to coash for tat to coash for tat to coash for tat to coash for tat to coash for tat to coash for ta
